# So you want to know about caches

In [1]:
import torch
import triton
import triton.language as tl
torch.set_default_device("cuda")

import ncu_magic

In [3]:
%%ncu_profile --labels add_k,add_k_mod
@triton.jit
def add_k_mod(x_ptr, y_ptr, out_ptr, N, BLOCK_SIZE: tl.constexpr):
    """`out = x + y`
    
    Each element is accessed only _once_. Therefore we don't want to pollute
    our cache with the loaded & stored elements.

    We can use the cache hints `.cs` to hint that we won't be reading
    this data again.
    """
    pid = tl.program_id(0)
    off = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = off < N

    x = tl.load(x_ptr + off, mask=mask, cache_modifier='.cv', volatile=True)
    y = tl.load(y_ptr + off, mask=mask, cache_modifier='.cv', volatile=True)
    out = x + y
    tl.store(out_ptr + off, out, mask=mask, cache_modifier='.cs')

@triton.jit
def add_k(x_ptr, y_ptr, out_ptr, N, BLOCK_SIZE: tl.constexpr):
    """`out = x + y`
    
    Each element is accessed only _once_. Therefore we don't want to pollute
    our cache with the loaded & stored elements.

    We can use the cache hints `.cs` to hint that we won't be reading
    this data again.
    """
    pid = tl.program_id(0)
    off = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = off < N

    x = tl.load(x_ptr + off, mask=mask)
    y = tl.load(y_ptr + off, mask=mask)
    out = x + y
    tl.store(out_ptr + off, out, mask=mask, cache_modifier='.cs')


N, BS = 10, 4
x = torch.arange(0, N)
y = torch.arange(0, N) + 1
out = torch.empty(N, dtype=x.dtype)
nvtx_launch("add_k", lambda: add_k[(triton.cdiv(N, BS),)](x, y, out, N, BS))
nvtx_launch("add_k_mod", lambda: add_k_mod[(triton.cdiv(N, BS),)](x, y, out, N, BS))

,time_ms,dram_read_GB,dram_write_GB,l2_read_GB,l1_glb_load_hit_rate,l2_pct_of_peak,dram_pct_of_peak
add_k,0.004160,0.000003,0.0,0.000024,0.75,0.34,0.05
add_k_mod,0.003424,0.000003,0.0,0.000024,0.00,0.67,0.05


In [ ]:
@triton.jit
def add_k_mod(x_ptr, y_ptr, out_ptr, N, BLOCK_SIZE: tl.constexpr):
    """`out = x + y`
    
    Each element is accessed only _once_. Therefore we don't want to pollute
    our cache with the loaded & stored elements.

    We can use the cache hints `.cs` to hint that we won't be reading
    this data again.
    """
    pid = tl.program_id(0)
    off = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = off < N

    x = tl.load(x_ptr + off, mask=mask, cache_modifier='.cv', volatile=True)
    y = tl.load(y_ptr + off, mask=mask, cache_modifier='.cv', volatile=True)
    out = x + y
    tl.store(out_ptr + off, out, mask=mask, cache_modifier='.cs', volatile=True)

add_k_mod[(triton.cdiv(N, BS),)](x, y, out, N, BS)

The `.cv` means volatile in the traditional sense: data can change unexpectedly, cache will give stale results, and don't cache new data since it will change soon.

- TODO: that's not the way we are using it, do we even need `.cv`???

Note this is (somewhat confusingly) different from `volatile=True`!

> So what's the difference between `.cv` and `.cs`? Why can't they be the same

We also don't need to add a cache eviction policty (`evict_policy` covered later) since our data isn't resident in cache.

## `cache_modifier`

Declare the cache level of your `load` or `store`.

### Loading

- `""`: Compiler managed, usually allocate in both L1 & L2
- `.ca`: `A`ll levels, aggressively target both L1 & L2.
  - Repeated access within _the same SM_ (for L1), maximize L1 hit rate.
  - Easily leads to L1 thrashing, small data.
- `.cg`: `G`lobal cache, L2 & HBM.
  - Primary goal is L1 thrashing mitigation. If tensor is too large to reside in L1 capacity of single SM, repeated loading will cause L1 thrashing.
- `.cv`

| Modifier | Target | Notes |
| -------- | ------ | ----- |
| `""`<br>(default) | L1 & L2 | Compiler managed |
| `.ca`<br>All levels | L1 & L2 | Cache aggressively across all levels.<br>Repeated access, good for small data with no risk of L1 thrashing. |
|  `.cg`<br>


[cache](https://docs.nvidia.com/cuda/parallel-thread-execution/index.html#cache-operators)